In [34]:
# ================================================
# 骨折分类 - 稳定可用最终版（已修复结构 + 可直接使用）
# ================================================
from google.colab import drive
drive.mount('/content/drive')

import os
import torch
from torchvision import transforms
from torchvision.datasets import ImageFolder
import timm
import gradio as gr
from PIL import Image
import torch.nn.functional as F
import warnings
warnings.filterwarnings('ignore')

print("正在连接 Drive 并加载数据...")

# 数据路径（已确认结构正确）
LOCAL_DATA = "/content/bone_data"

# 加载类别
full_ds = ImageFolder(LOCAL_DATA)
class_names = full_ds.classes
NUM_CLASSES = len(class_names)

print(f"✅ 成功加载 {NUM_CLASSES} 类骨折")
print("类别:", class_names)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 加载模型（优先使用训练好的模型，如果没有则用预训练模型）
model = timm.create_model("convnextv2_nano.fcmae_ft_in22k_in1k", pretrained=False, num_classes=NUM_CLASSES)

try:
    model.load_state_dict(torch.load("/content/best_bone_model.pth", weights_only=True, map_location=DEVICE))
    print("✅ 成功加载之前训练的最佳模型！")
except:
    print("⚠️ 未找到训练模型，使用预训练模型...")
    model = timm.create_model("convnextv2_nano.fcmae_ft_in22k_in1k", pretrained=True, num_classes=NUM_CLASSES)

model = model.to(DEVICE)
model.eval()

print("✅ 模型准备就绪！")

# 预测函数
def predict(image):
    if image is None:
        return "请上传一张骨折 X 光图片", {}

    tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    x = tf(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(x)
        probs = F.softmax(logits, dim=1)[0].cpu().numpy()

    pred_idx = int(probs.argmax())
    confidence = float(probs[pred_idx] * 100)

    top3_idx = probs.argsort()[-3:][::-1]
    top3 = {class_names[i]: round(float(probs[i]*100), 2) for i in top3_idx}

    emoji = "🟢" if confidence > 80 else "🟡" if confidence > 60 else "🔴"
    result_text = f"{emoji} **预测结果：{class_names[pred_idx]}**\n置信度：**{confidence:.1f}%**"

    return result_text, top3

# 启动网页
print("\n🚀 正在启动网页...")

iface = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil", label="上传骨折 X 光图片"),
    outputs=[
        gr.Markdown(label="AI 预测结果"),
        gr.Label(label="Top 3 预测概率")
    ],
    title="🦴 骨折类型智能分类系统",
    description="上传骨折 X 光图片，AI 自动判断类型",
    theme=gr.themes.Soft(),
    allow_flagging="never"
)

iface.launch(share=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
正在连接 Drive 并加载数据...
✅ 成功加载 12 类骨折
类别: ['Avulsion fracture', 'Comminuted fracture', 'Compression-Crush fracture', 'Fracture Dislocation', 'Greenstick fracture', 'Hairline Fracture', 'Impacted fracture', 'Intra-articular fracture', 'Longitudinal fracture', 'Oblique fracture', 'Pathological fracture', 'Spiral Fracture']
⚠️ 未找到训练模型，使用预训练模型...


✅ 模型准备就绪！

🚀 正在启动网页...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d147b8dc95d5ba1474.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [35]:
# ================================================
# 提高 F1-score 训练版（推荐运行）
# ================================================
from google.colab import drive
drive.mount('/content/drive')

import os, time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchvision import transforms
from torchvision.datasets import ImageFolder
import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("正在准备数据...")

LOCAL_DATA = "/content/bone_data"
full_ds = ImageFolder(LOCAL_DATA)
class_names = full_ds.classes
NUM_CLASSES = len(class_names)

# 强数据增强（针对 X 光图像）
train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# 分割数据集 + 加权采样（解决类别不平衡）
indices = list(range(len(full_ds)))
train_idx, val_idx = train_test_split(indices, test_size=0.2, stratify=full_ds.targets, random_state=42)

train_dataset = Subset(ImageFolder(LOCAL_DATA, transform=train_tf), train_idx)
val_dataset   = Subset(ImageFolder(LOCAL_DATA, transform=eval_tf), val_idx)

# 加权采样
targets = np.array([full_ds.targets[i] for i in train_idx])
class_sample_count = np.bincount(targets)
weight = 1. / class_sample_count
samples_weight = weight[targets]
sampler = WeightedRandomSampler(torch.from_numpy(samples_weight), len(samples_weight))

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"训练集 {len(train_dataset)} 张 | 验证集 {len(val_dataset)} 张")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 使用 Focal Loss（对难样本更好）
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=1.0):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
    def forward(self, logits, targets):
        ce_loss = nn.CrossEntropyLoss()(logits, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

# 模型
model = timm.create_model("convnextv2_nano.fcmae_ft_in22k_in1k", pretrained=True, num_classes=NUM_CLASSES)
model = model.to(DEVICE)

criterion = FocalLoss(gamma=2.0)
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

print("开始训练（目标提高 F1-score）...")

best_f1 = 0.0
patience = 8
pat = 0

for epoch in range(1, 21):
    # 训练
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    # 验证
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    current_f1 = f1_score(all_labels, all_preds, average='macro') * 100
    print(f"Epoch {epoch:2d} | Val F1-macro: {current_f1:.2f}%")

    if current_f1 > best_f1:
        best_f1 = current_f1
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        pat = 0
        print(f"  → 新最佳 F1: {best_f1:.2f}% ✅")
    else:
        pat += 1

    scheduler.step()

    if pat >= patience:
        print("早停触发")
        break

# 保存最佳模型
model.load_state_dict(best_state)
torch.save(best_state, "/content/best_bone_model.pth")
print(f"\n🎉 训练完成！最佳验证 F1-score: {best_f1:.2f}%")
print("💾 最佳模型已保存到 /content/best_bone_model.pth")

# ==================== 训练完立即启动网页 ====================
print("\n🚀 启动网页（使用刚训练好的模型）...")

def predict(image):
    if image is None:
        return "请上传图片", {}

    tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    x = tf(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(x)
        probs = F.softmax(logits, dim=1)[0].cpu().numpy()

    pred_idx = int(probs.argmax())
    confidence = float(probs[pred_idx] * 100)

    top3_idx = probs.argsort()[-3:][::-1]
    top3 = {class_names[i]: round(float(probs[i]*100), 2) for i in top3_idx}

    emoji = "🟢" if confidence > 80 else "🟡" if confidence > 60 else "🔴"
    result_text = f"{emoji} **预测结果：{class_names[pred_idx]}**\n置信度：**{confidence:.1f}%**"

    return result_text, top3

iface = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil", label="上传骨折 X 光图片"),
    outputs=[gr.Markdown(label="预测结果"), gr.Label(label="Top 3 概率")],
    title="🦴 骨折类型智能分类系统 - 高 F1 优化版",
    description=f"已训练优化 | 当前最佳验证 F1: {best_f1:.2f}%",
    theme=gr.themes.Soft(),
    allow_flagging="never"
)

iface.launch(share=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
正在准备数据...
训练集 1348 张 | 验证集 338 张
开始训练（目标提高 F1-score）...
Epoch  1 | Val F1-macro: 4.83%
  → 新最佳 F1: 4.83% ✅
Epoch  2 | Val F1-macro: 7.47%
  → 新最佳 F1: 7.47% ✅
Epoch  3 | Val F1-macro: 22.07%
  → 新最佳 F1: 22.07% ✅
Epoch  4 | Val F1-macro: 23.16%
  → 新最佳 F1: 23.16% ✅
Epoch  5 | Val F1-macro: 25.23%
  → 新最佳 F1: 25.23% ✅
Epoch  6 | Val F1-macro: 30.08%
  → 新最佳 F1: 30.08% ✅
Epoch  7 | Val F1-macro: 33.13%
  → 新最佳 F1: 33.13% ✅
Epoch  8 | Val F1-macro: 40.53%
  → 新最佳 F1: 40.53% ✅
Epoch  9 | Val F1-macro: 42.70%
  → 新最佳 F1: 42.70% ✅
Epoch 10 | Val F1-macro: 42.81%
  → 新最佳 F1: 42.81% ✅
Epoch 11 | Val F1-macro: 44.86%
  → 新最佳 F1: 44.86% ✅
Epoch 12 | Val F1-macro: 47.26%
  → 新最佳 F1: 47.26% ✅
Epoch 13 | Val F1-macro: 50.81%
  → 新最佳 F1: 50.81% ✅
Epoch 14 | Val F1-macro: 52.52%
  → 新最佳 F1: 52.52% ✅
Epoch 15 | Val F1-macro: 50.46%
Epoch 16 | Val F1-macro: 50.29%
Epoch 17 | Va